In [1]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
BUCKET = "gs://retailsmart-lake-retailsmart-bd"

spark.conf.set("spark.sql.catalog.retailsmart", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.retailsmart.type", "hadoop")
spark.conf.set("spark.sql.catalog.retailsmart.warehouse", f"{BUCKET}/lakehouse")
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.sql("CREATE NAMESPACE IF NOT EXISTS retailsmart.universal")

DataFrame[]

In [2]:
raw = (spark.read.option("header", True).option("inferSchema", True)
       .csv(f"{BUCKET}/landing/uci/online_retail.csv"))

df = (raw
  .withColumn("factura", F.col("InvoiceNo").cast("string"))
  .withColumn("codigo_producto", F.col("StockCode").cast("string"))
  .withColumnRenamed("Description", "descripcion")
  .withColumnRenamed("Quantity", "cantidad")
  .withColumnRenamed("InvoiceDate", "fecha_factura")
  .withColumnRenamed("UnitPrice", "precio_unitario")
  .withColumnRenamed("Country", "pais")
  .withColumn("cliente_id", F.col("CustomerID").cast("long").cast("string"))
  .withColumn("es_cancelacion", F.col("factura").startswith("C"))
  .drop("InvoiceNo", "StockCode", "CustomerID"))
df.printSchema()

root
 |-- descripcion: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- fecha_factura: timestamp (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- pais: string (nullable = true)
 |-- factura: string (nullable = true)
 |-- codigo_producto: string (nullable = true)
 |-- cliente_id: string (nullable = true)
 |-- es_cancelacion: boolean (nullable = true)



In [3]:
reglas = [
  ("factura nula", F.col("factura").isNull()),
  ("código de producto nulo", F.col("codigo_producto").isNull()),
  ("fecha de factura nula", F.col("fecha_factura").isNull()),
  ("precio negativo", F.col("precio_unitario") < 0),
  ("precio cero", F.col("precio_unitario") == 0),
  ("cantidad no positiva fuera de cancelación",
   (~F.col("es_cancelacion")) & (F.col("cantidad") <= 0)),
]
checked = df.withColumn("motivo", F.coalesce(*[F.when(c, F.lit(t)) for t, c in reglas]))
malos = checked.filter(F.col("motivo").isNotNull())
buenos = checked.filter(F.col("motivo").isNull()).drop("motivo").dropDuplicates()

meta = lambda d: (d.withColumn("_fuente", F.lit("uci"))
                   .withColumn("_cargado_en", F.current_timestamp()))
t_ok = "retailsmart.universal.silver_tienda_ventas_v1"
t_bad = "retailsmart.universal.cuarentena_tienda_ventas_v1"
meta(buenos).writeTo(t_ok).using("iceberg").createOrReplace()
meta(malos).writeTo(t_bad).using("iceberg").createOrReplace()

n_raw, n_ok, n_bad = raw.count(), spark.table(t_ok).count(), spark.table(t_bad).count()
print(f"ventas UCI: leídas {n_raw} | válidas {n_ok} | rechazadas {n_bad} | duplicados {n_raw - n_ok - n_bad}")
spark.table(t_bad).groupBy("motivo").count().show(truncate=False)
spark.sql(f"SELECT es_cancelacion, cliente_id IS NULL AS sin_cliente, COUNT(*) AS filas FROM {t_ok} GROUP BY 1, 2 ORDER BY 1, 2").show()

ventas UCI: leídas 541909 | válidas 534129 | rechazadas 2517 | duplicados 5263


+---------------+-----+
|motivo         |count|
+---------------+-----+
|precio negativo|2    |
|precio cero    |2515 |
+---------------+-----+



+--------------+-----------+------+
|es_cancelacion|sin_cliente| filas|
+--------------+-----------+------+
|         false|      false|392692|
|         false|       true|132186|
|          true|      false|  8872|
|          true|       true|   379|
+--------------+-----------+------+

